# Boston Community Intervention — Crew & Catalyst Network Dashboard

Interactive map matching the KC StoryMap visual style, adapted for Boston data.

### Data files (place in same directory as notebook)
- `Fake_Small_BostonCrews_Networks.xlsx` — crew/catalyst data
- `2020_Census_Tracts_in_Boston.csv` — census tract IDs

### Requirements
```
pip install pandas folium requests branca shapely openpyxl geopandas
```

### Features
- Green-to-red choropleth by census tract, **live-filtered by date range**
- Date-range widget (start + end inputs) + **YTD button** — client-side, no re-run needed
- **Stipend** (teal) vs **In-Network** (mauve) catalyst dots
- Crew popups show catalysts grouped by type, with status and assigned outreach worker
- External crew refs (e.g. 'Capetown') handled gracefully — logged, not crashed

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import json
import random
import math
import requests
from shapely.geometry import Point
from datetime import datetime

## 1. Scrape Shooting Data from Boston PD

In [2]:
USE_LOCAL = False
LOCAL_CSV = "boston_shootings_latest.csv"

SHOOTINGS_CSV_URL = (
    "https://data.boston.gov/dataset/"
    "e63a37e1-be79-4722-89e6-9e7e2a3da6d1/resource/"
    "73c7e069-701f-4910-986d-b950f46c91a1/download/tmplstdidm9.csv"
)

if not USE_LOCAL:
    print("Downloading latest shooting data from Analyze Boston ...")
    try:
        resp = requests.get(SHOOTINGS_CSV_URL, timeout=60)
        resp.raise_for_status()
        with open(LOCAL_CSV, "wb") as f:
            f.write(resp.content)
        print(f"  Saved to {LOCAL_CSV}")
    except Exception as e:
        print(f"  Download failed: {e} -- falling back to local CSV")

df_shoot = pd.read_csv(LOCAL_CSV)
df_shoot["shooting_date"] = pd.to_datetime(df_shoot["shooting_date"], utc=True)

print(f"Loaded {len(df_shoot):,} incidents")
print(f"Date range: {df_shoot['shooting_date'].min().strftime('%Y-%m-%d')} to "
      f"{df_shoot['shooting_date'].max().strftime('%Y-%m-%d')}")
print(f"Fatal: {(df_shoot['shooting_type_v2']=='Fatal').sum():,}  |  "
      f"Non-Fatal: {(df_shoot['shooting_type_v2']=='Non-Fatal').sum():,}")

  Saved to boston_shootings_latest.csv
Loaded 2,181 incidents
Date range: 2015-01-01 to 2026-04-02
Fatal: 361  |  Non-Fatal: 1,820


## 2. Load Crew & Catalyst Network

**Schema** (one row per crew, multi-value cells use actual newline `\n` as delimiter):

| Column | Notes |
|---|---|
| `Crew` | Crew name |
| `Lat`, `Long` | Crew map location |
| `Catalysts` | `\n`-separated catalyst names |
| `Catalyst Type` | `\n`-separated, positionally aligned: `Stipend` or `Network` |
| `Catalyst Status` | `\n`-separated active/inactive flags |
| `Assigned to` | `\n`-separated outreach worker per catalyst |
| `Catalst Network` | `\n`-separated **catalyst names** (individual-level network links) |
| `Affiliations/Collaborations` | `\n`-separated crew names (lowercase, may be external) |
| `Conflicts w Other Crews` | `\n`-separated crew names (lowercase, may be external) |

In [3]:
CREW_PATH = "Fake_Small_BostonCrews_Networks.xlsx"

df_crews = pd.read_excel(CREW_PATH)
df_crews.columns = df_crews.columns.str.strip()
df_crews["Crew"] = df_crews["Crew"].str.strip().str.title()

# Actual in-cell delimiter is newline (\n), NOT comma or double-space
def split_cell(val):
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).split("\n") if x.strip()]

# Case-insensitive crew resolver
# Affiliations/conflicts use lowercase shorthand (e.g. 'cod' -> 'Cod')
# External crews (e.g. 'Capetown') resolve to None -- handled gracefully
crew_names_lower = {row["Crew"].lower(): row["Crew"]
                    for _, row in df_crews.iterrows()}
def resolve_crew(name):
    return crew_names_lower.get(name.strip().lower())

# Per-catalyst lookups
catalyst_to_crew      = {}  # cat_name -> (crew, lat, lon)
catalyst_type_map     = {}  # cat_name -> 'stipend' | 'network' | 'unknown'
catalyst_status_map   = {}  # cat_name -> status string
catalyst_assigned_map = {}  # cat_name -> assigned outreach worker
crew_catalysts        = {}  # crew -> [cat_names]

for _, row in df_crews.iterrows():
    crew     = row["Crew"]
    cats     = split_cell(row["Catalysts"])
    types    = split_cell(row["Catalyst Type"])
    statuses = split_cell(row["Catalyst Status"])
    assigned = split_cell(row["Assigned to"])

    n = len(cats)
    types    += ["unknown"] * max(0, n - len(types))
    statuses += [""]        * max(0, n - len(statuses))
    assigned += [""]        * max(0, n - len(assigned))

    crew_catalysts[crew] = cats

    for cat, ctype, cstatus, cassigned in zip(cats, types, statuses, assigned):
        catalyst_to_crew[cat]      = (crew, row["Lat"], row["Long"])
        catalyst_status_map[cat]   = cstatus
        catalyst_assigned_map[cat] = cassigned
        t = ctype.strip().lower()
        catalyst_type_map[cat] = (
            "stipend" if "stip" in t else
            "network" if "net"  in t else
            "unknown"
        )

# Crew-to-crew relationships
crew_affiliations = []
crew_conflicts    = []
external_refs     = []  # crews mentioned but not in dataset

for _, row in df_crews.iterrows():
    crew = row["Crew"]
    for other in split_cell(row["Affiliations/Collaborations"]):
        r = resolve_crew(other)
        if r and r != crew:
            pair = tuple(sorted([crew, r]))
            if pair not in crew_affiliations:
                crew_affiliations.append(pair)
        elif not r:
            external_refs.append((crew, "affiliation", other))

    for other in split_cell(row["Conflicts w Other Crews"]):
        r = resolve_crew(other)
        if r and r != crew:
            pair = tuple(sorted([crew, r]))
            if pair not in crew_conflicts:
                crew_conflicts.append(pair)
        elif not r:
            external_refs.append((crew, "conflict", other))

# Catalyst-to-catalyst network edges
# Reading rule for 'Catalst Network' column:
#   - Each cell line is split by \n
#   - If a line has 2+ comma-separated names -> pairwise links between ALL of them
#     e.g. 'Tairby Brown, Ariah Peterson' = one edge: Tairby Brown <-> Ariah Peterson
#   - If a line has 1 name -> link from this crew's first catalyst to that person
#     e.g. Bush Gardens 'DJ' = Justin Brown <-> DJ
# Fuzzy matching handles minor typos (e.g. 'Michael B' vs 'Micheal B')

def fuzzy_resolve_catalyst(name, all_catalysts):
    """Exact match first, then case-insensitive, then prefix match."""
    if name in all_catalysts:
        return name
    name_lower = name.lower().strip()
    for c in all_catalysts:
        if c.lower().strip() == name_lower:
            return c
    # Prefix match: 'Michael B' matches 'Micheal B' or vice versa by first token
    name_parts = set(name_lower.split())
    for c in all_catalysts:
        c_parts = set(c.lower().split())
        if len(name_parts & c_parts) >= max(1, len(name_parts) - 1):
            return c
    return None

all_catalysts = set(catalyst_to_crew.keys())
catalyst_edges = []
unresolved_net = []  # names that couldn't be matched

for _, row in df_crews.iterrows():
    crew     = row["Crew"]
    src_cats = crew_catalysts.get(crew, [])
    lines    = split_cell(row["Catalst Network"])

    for line in lines:
        names_raw = [x.strip() for x in line.split(",") if x.strip()]
        # Resolve each name (with fuzzy fallback)
        resolved = []
        for n in names_raw:
            r = fuzzy_resolve_catalyst(n, all_catalysts)
            if r:
                resolved.append(r)
            else:
                unresolved_net.append((crew, n))

        if len(resolved) >= 2:
            # Pairwise links between all names on this line
            for i in range(len(resolved)):
                for j in range(i + 1, len(resolved)):
                    a, b = resolved[i], resolved[j]
                    if a != b:
                        pair = tuple(sorted([a, b]))
                        if pair not in catalyst_edges:
                            catalyst_edges.append(pair)
        elif len(resolved) == 1 and src_cats:
            # Single name: link from this crew's first catalyst to the target
            tgt = resolved[0]
            for src in src_cats:
                if src != tgt:
                    pair = tuple(sorted([src, tgt]))
                    if pair not in catalyst_edges:
                        catalyst_edges.append(pair)
                    break

# Diagnostic summary
stipend_count = sum(1 for v in catalyst_type_map.values() if v == "stipend")
network_count = sum(1 for v in catalyst_type_map.values() if v == "network")
unknown_count = sum(1 for v in catalyst_type_map.values() if v == "unknown")

print("Crews & Catalysts")
print(f"  Crews         : {len(df_crews)}")
print(f"  Catalysts     : {len(catalyst_to_crew)}  "
      f"(stipend={stipend_count}, network={network_count}, unknown={unknown_count})")
print(f"  Affiliations  : {len(crew_affiliations)}")
print(f"  Conflicts     : {len(crew_conflicts)}")
print(f"  Catalyst links: {len(catalyst_edges)}")
if external_refs:
    print(f"\n  External crew refs (not in dataset -- arcs skipped):")
    for c, rel, name in external_refs:
        print(f"    {c} [{rel}] -> '{name}'")
if unresolved_net:
    print(f"\n  Unresolved catalyst network names (check spelling in 'Catalst Network' col):")
    for crew, name in unresolved_net:
        print(f"    {crew} -> '{name}'")
if unknown_count > 0:
    print(f"\n  WARNING: {unknown_count} catalyst(s) have unrecognized type:")
    for cat, t in catalyst_type_map.items():
        if t == "unknown":
            print(f"    '{cat}' -- check 'Catalyst Type' column")

Crews & Catalysts
  Crews         : 7
  Catalysts     : 28  (stipend=15, network=13, unknown=0)
  Affiliations  : 6
  Conflicts     : 3
  Catalyst links: 15

  External crew refs (not in dataset -- arcs skipped):
    Bush Gardens [conflict] -> 'Capetown'
    Boom [affiliation] -> 'Capetown'
    Hot Tamales [conflict] -> 'Capetown'

  Unresolved catalyst network names (check spelling in 'Catalst Network' col):
    Boom -> 'Jeremy'


## 3. Load 2020 Census Tract Boundaries

In [4]:
df_tracts_csv = pd.read_csv("2020_Census_Tracts_in_Boston.csv")
boston_geoids = set(df_tracts_csv["geoid20"].astype(str))
print(f"Boston tract IDs from CSV: {len(boston_geoids)}")

TIGER_URL = (
    "https://www2.census.gov/geo/tiger/GENZ2020/shp/"
    "cb_2020_25_tract_500k.zip"
)

print("Downloading Census Bureau tract polygons for Massachusetts ...")
gdf_all = gpd.read_file(TIGER_URL)
print(f"  Total MA tracts: {len(gdf_all)}")

gdf_tracts = gdf_all[gdf_all["COUNTYFP"] == "025"].copy()
if len(gdf_tracts) == 0:
    gdf_tracts = gdf_all[gdf_all["GEOID"].isin(boston_geoids)].copy()

gdf_tracts["tract_id"] = gdf_tracts["GEOID"].astype(str)
gdf_tracts = gdf_tracts.to_crs("EPSG:4326")
print(f"  Boston tracts matched: {len(gdf_tracts)}")

Boston tract IDs from CSV: 207
  Total MA tracts: 1613
  Boston tracts matched: 234


## 4. Prepare Shooting Data for Client-Side Filtering

All incident records are embedded as a JSON blob in the exported HTML.
The date widget re-filters and repaints the choropleth entirely client-side.
The weekly `papermill` refresh regenerates fresh HTML with updated data.

In [5]:
DISTRICT_COORDS = {
    "A1":  (42.3580, -71.0580), "A7":  (42.3690, -71.0390),
    "A15": (42.3270, -71.0470), "B2":  (42.3290, -71.0840),
    "B3":  (42.3010, -71.0800), "C6":  (42.3380, -71.0380),
    "C11": (42.2990, -71.0610), "D4":  (42.3440, -71.0770),
    "D14": (42.3530, -71.1320), "E5":  (42.3050, -71.1150),
    "E13": (42.3230, -71.1040), "E18": (42.2830, -71.1520),
}

random.seed(42)
shoot_records = []

for _, row in df_shoot.iterrows():
    dist = row["district"]
    if dist not in DISTRICT_COORDS:
        continue
    lat, lon = DISTRICT_COORDS[dist]
    shoot_records.append({
        "lat":      round(lat + random.gauss(0, 0.008), 6),
        "lon":      round(lon + random.gauss(0, 0.009), 6),
        "date":     row["shooting_date"].strftime("%Y-%m-%d"),
        "fatal":    int(row["shooting_type_v2"] == "Fatal"),
        "district": dist,
    })

print(f"Shoot records for JS embed: {len(shoot_records):,}")

# Full-dataset tract counts for initial page load
gdf_points_all = gpd.GeoDataFrame(
    [{"geometry": Point(r["lon"], r["lat"]), "incident": 1} for r in shoot_records],
    crs="EPSG:4326"
)
joined_all = gpd.sjoin(
    gdf_points_all, gdf_tracts[["tract_id", "geometry"]],
    how="left", predicate="within"
)
tract_counts_all = joined_all.groupby("tract_id")["incident"].sum()
gdf_tracts["shoot_count"] = (
    gdf_tracts["tract_id"].map(tract_counts_all).fillna(0).astype(int)
)

# Simplified tract polygons for JS ray-casting point-in-polygon
tract_polygons_for_js = []
for _, tr in gdf_tracts.iterrows():
    geom = tr["geometry"].simplify(0.001, preserve_topology=True)
    if geom.geom_type == "Polygon":
        coords = list(geom.exterior.coords)
    else:
        coords = list(max(geom.geoms, key=lambda g: g.area).exterior.coords)
    tract_polygons_for_js.append({
        "id":     tr["tract_id"],
        "coords": [[round(c[1], 5), round(c[0], 5)] for c in coords],
    })

print(f"Tract polygons embedded: {len(tract_polygons_for_js)}")
print(f"Initial counts -- max per tract: {gdf_tracts['shoot_count'].max()}, "
      f"total: {int(gdf_tracts['shoot_count'].sum()):,}")

Shoot records for JS embed: 2,181
Tract polygons embedded: 234
Initial counts -- max per tract: 91, total: 2,101


## 5. Build the Map

In [6]:
# Color constants (Okabe-Ito colorblind-safe palette)
CREW_ORANGE   = "#E8782A"  # Crew circle fill
STIPEND_COLOR = "#009E73"  # Teal  -- stipend catalysts
NETWORK_COLOR = "#CC79A7"  # Mauve -- in-network catalysts
UNKNOWN_COLOR = "#999999"  # Grey fallback

def cat_color(cat_name):
    t = catalyst_type_map.get(cat_name, "unknown")
    return {"stipend": STIPEND_COLOR, "network": NETWORK_COLOR}.get(t, UNKNOWN_COLOR)

def cat_type_label(cat_name):
    t = catalyst_type_map.get(cat_name, "unknown")
    return {"stipend": "Stipend", "network": "In-Network"}.get(t, "Unclassified")

# Base map
m = folium.Map(location=[42.31, -71.08], zoom_start=12, tiles="CartoDB positron")

# Choropleth via GeoJson + custom style_function (avoids bin-range ValueError)
tracts_geojson = json.loads(gdf_tracts.to_json())
for i, feat in enumerate(tracts_geojson["features"]):
    feat["id"] = gdf_tracts.iloc[i]["tract_id"]
    feat["properties"]["shoot_count"] = int(gdf_tracts.iloc[i]["shoot_count"])
    feat["properties"]["tract_id"]    = gdf_tracts.iloc[i]["tract_id"]

def count_to_color(count):
    if count == 0:     return "#2d8c4e"
    elif count == 1:   return "#78b85e"
    elif count <= 3:   return "#c8d96f"
    elif count <= 10:  return "#ffe08b"
    elif count <= 25:  return "#fdbe6f"
    elif count <= 50:  return "#f4845f"
    elif count <= 100: return "#e8533e"
    else:              return "#c0233c"

def style_function(feature):
    count = feature["properties"].get("shoot_count", 0)
    return {"fillColor": count_to_color(count), "fillOpacity": 0.70,
            "color": "#bbb", "weight": 0.8}

geojson_layer = folium.GeoJson(
    tracts_geojson,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["tract_id", "shoot_count"],
        aliases=["Census Tract:", "Shootings (filtered):"],
        style="font-family:'Helvetica Neue',system-ui,sans-serif;font-size:14px;",
    ),
    name="Shootings by census tract",
    overlay=True,
)
geojson_layer.add_to(m)
CHOROPLETH_LAYER_ID = geojson_layer.get_name()
print(f"Choropleth: {len(tracts_geojson['features'])} tracts")

Choropleth: 234 tracts


In [7]:
# Crew-to-crew affiliations (solid sky blue)
affiliations_fg = folium.FeatureGroup(name="Affiliations", show=True)
crew_loc = {row["Crew"]: (row["Lat"], row["Long"]) for _, row in df_crews.iterrows()}

for a, b in crew_affiliations:
    if a in crew_loc and b in crew_loc:
        folium.PolyLine(
            locations=[crew_loc[a], crew_loc[b]],
            color="#56B4E9", weight=2.5, opacity=0.7,
            tooltip=f"Affiliation: {a} <-> {b}",
        ).add_to(affiliations_fg)
affiliations_fg.add_to(m)

# Crew-to-crew conflicts (dashed vermilion)
conflicts_fg = folium.FeatureGroup(name="Conflicts", show=True)
for a, b in crew_conflicts:
    if a in crew_loc and b in crew_loc:
        folium.PolyLine(
            locations=[crew_loc[a], crew_loc[b]],
            color="#D55E00", weight=2.5, opacity=0.7, dash_array="10,6",
            tooltip=f"Conflict: {a} <-> {b}",
        ).add_to(conflicts_fg)
conflicts_fg.add_to(m)

# Catalyst-to-catalyst links (dotted grey)
catalyst_net_fg = folium.FeatureGroup(name="Catalyst links", show=True)

def jitter(lat, lon, seed_key, radius=0.003):
    rnd = random.Random(seed_key)
    angle = rnd.uniform(0, 2 * math.pi)
    r = rnd.uniform(0.3, 1.0) * radius
    return (lat + r * math.cos(angle), lon + r * math.sin(angle))

catalyst_loc = {}
for cat, (crew, lat, lon) in catalyst_to_crew.items():
    catalyst_loc[cat] = jitter(lat, lon, cat)

for a, b in catalyst_edges:
    if a in catalyst_loc and b in catalyst_loc:
        folium.PolyLine(
            locations=[catalyst_loc[a], catalyst_loc[b]],
            color="#888", weight=1.2, opacity=0.4, dash_array="2,5",
            tooltip=f"Network: {a} <-> {b}",
        ).add_to(catalyst_net_fg)
catalyst_net_fg.add_to(m)

print(f"Arcs: {len(crew_affiliations)} affiliations, "
      f"{len(crew_conflicts)} conflicts, {len(catalyst_edges)} catalyst links")

Arcs: 6 affiliations, 3 conflicts, 15 catalyst links


In [8]:
# Crew markers + individual catalyst dots
crews_fg = folium.FeatureGroup(name="Crews & catalysts", show=True)

for _, row in df_crews.iterrows():
    crew   = row["Crew"]
    cats   = crew_catalysts.get(crew, [])
    n_cats = len(cats)

    if n_cats > 12:   radius = 22
    elif n_cats >= 9: radius = 18
    elif n_cats >= 6: radius = 14
    elif n_cats >= 4: radius = 10
    else:             radius = 6

    stipend_cats = [c for c in cats if catalyst_type_map.get(c) == "stipend"]
    network_cats = [c for c in cats if catalyst_type_map.get(c) == "network"]
    other_cats   = [c for c in cats if catalyst_type_map.get(c) == "unknown"]

    def cat_rows(cat_list, color, label):
        if not cat_list:
            return ""
        rows = "".join(
            f'<tr>'
            f'<td style="padding:3px 10px 3px 0;color:#333;font-size:13px;">{c}</td>'
            f'<td style="padding:3px 10px 3px 0;color:#666;font-size:12px;">{catalyst_status_map.get(c,"")}</td>'
            f'<td style="padding:3px 0;color:#888;font-size:12px;">{catalyst_assigned_map.get(c,"")}</td>'
            f'</tr>'
            for c in cat_list
        )
        return (
            f'<tr><td colspan="3" style="padding-top:9px;font-weight:700;'
            f'color:{color};font-size:13px;">{label}</td></tr>'
            + rows
        )

    popup_html = (
        f'<div style="font-family:\'Helvetica Neue\',system-ui,sans-serif;min-width:300px;max-width:380px;">'
        f'<div style="font-size:16px;font-weight:bold;color:{CREW_ORANGE};margin-bottom:3px;">{crew}</div>'
        f'<div style="font-size:13px;color:#555;margin-bottom:10px;">{n_cats} catalyst(s)</div>'
        f'<table style="font-size:13px;border-collapse:collapse;width:100%;">'
        f'<tr style="border-bottom:1px solid #eee;">'
        f'<th style="text-align:left;font-size:11px;color:#aaa;padding-bottom:4px;">Name</th>'
        f'<th style="text-align:left;font-size:11px;color:#aaa;">Status</th>'
        f'<th style="text-align:left;font-size:11px;color:#aaa;">Assigned to</th>'
        f'</tr>'
        + cat_rows(stipend_cats, STIPEND_COLOR, "Stipend")
        + cat_rows(network_cats, NETWORK_COLOR, "In-Network")
        + cat_rows(other_cats,   UNKNOWN_COLOR, "Unclassified")
        + '</table></div>'
    )

    folium.CircleMarker(
        location=[row["Lat"], row["Long"]],
        radius=radius,
        color="#c45a1a", weight=2,
        fill=True, fill_color=CREW_ORANGE, fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=400),
        tooltip=folium.Tooltip(
            f"{crew} ({n_cats} catalysts)",
            style=(
                "font-family:'Helvetica Neue',system-ui,sans-serif;"
                "font-size:14px;font-weight:600;"
                "background:rgba(255,255,255,0.97);"
                "border:1px solid #ddd;border-radius:4px;"
                "padding:5px 12px;box-shadow:0 2px 6px rgba(0,0,0,0.12);"
                "color:#222;"
            ),
        ),
    ).add_to(crews_fg)

    folium.Marker(
        location=[row["Lat"], row["Long"]],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:13px;color:#333;font-weight:700;'
                f'text-shadow:0 0 4px #fff, 0 0 8px #fff;'
                f'white-space:nowrap;transform:translate(18px,-7px);">{crew}</div>'
            ),
            icon_size=(160, 20), icon_anchor=(0, 0),
        ),
    ).add_to(crews_fg)

# Individual catalyst dots, colored by type
# Each dot gets a labeled popup card (shown on hover via tooltip=) and
# a richer click popup with all fields labeled.
for cat, (clat, clon) in catalyst_loc.items():
    color  = cat_color(cat)
    label  = cat_type_label(cat)
    status = catalyst_status_map.get(cat, "")
    worker = catalyst_assigned_map.get(cat, "")
    crew   = catalyst_to_crew[cat][0]

    # Color badge for type
    type_badge = (
        f'<span style="display:inline-block;padding:1px 7px;border-radius:10px;'
        f'background:{color};color:#fff;font-size:11px;font-weight:600;">'
        f'{label}</span>'
    )

    def field_row(col_label, value, val_color="#333"):
        if not value or value == "None Assigned":
            return ""
        return (
            f'<tr>'
            f'<td style="padding:3px 10px 3px 0;color:#888;font-size:12px;'
            f'white-space:nowrap;font-weight:500;">{col_label}</td>'
            f'<td style="padding:3px 0;color:{val_color};font-size:13px;'
            f'font-weight:500;">{value}</td>'
            f'</tr>'
        )

    cat_popup_html = (
        f'<div style="font-family:\'Helvetica Neue\',system-ui,sans-serif;'
        f'min-width:200px;max-width:260px;padding:2px;">'
        f'<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px;">'
        f'{cat}</div>'
        f'{type_badge}'
        f'<table style="border-collapse:collapse;width:100%;margin-top:8px;">'
        + field_row("Crew",        crew)
        + field_row("Status",      status)
        + field_row("Assigned to", worker)
        + f'</table></div>'
    )

    # Tooltip string: short labeled summary shown on hover
    tip = f"{cat}  ·  {label}  ·  {crew}"

    folium.CircleMarker(
        location=[clat, clon],
        radius=5,
        color=color, weight=1.2,
        fill=True, fill_color=color, fill_opacity=0.85,
        tooltip=folium.Tooltip(
            tip,
            style=(
                "font-family:'Helvetica Neue',system-ui,sans-serif;"
                "font-size:13px;font-weight:500;"
                "background:rgba(255,255,255,0.97);"
                "border:1px solid #ddd;border-radius:4px;"
                "padding:5px 10px;box-shadow:0 2px 6px rgba(0,0,0,0.12);"
                "color:#222;"
            ),
        ),
        popup=folium.Popup(cat_popup_html, max_width=280),
    ).add_to(crews_fg)

crews_fg.add_to(m)
print(f"Markers: {len(df_crews)} crews, {len(catalyst_loc)} catalysts "
      f"(stipend={stipend_count}, network={network_count})")

Markers: 7 crews, 28 catalysts (stipend=15, network=13)


## 6. Inject Date-Filter Widget + Live Choropleth JS

In [9]:
DATA_START = df_shoot["shooting_date"].min().strftime("%Y-%m-%d")
DATA_END   = df_shoot["shooting_date"].max().strftime("%Y-%m-%d")
TODAY      = datetime.today().strftime("%Y-%m-%d")
CUR_YEAR   = datetime.today().year
TOTAL_N    = len(shoot_records)

shoot_json  = json.dumps(shoot_records)
tracts_json = json.dumps(tract_polygons_for_js)

filter_js = f"""
<script>
var SHOOT_DATA  = {shoot_json};
var TRACT_POLYS = {tracts_json};

function countToColor(n) {{
    if (n === 0)  return '#2d8c4e';
    if (n === 1)  return '#78b85e';
    if (n <= 3)   return '#c8d96f';
    if (n <= 10)  return '#ffe08b';
    if (n <= 25)  return '#fdbe6f';
    if (n <= 50)  return '#f4845f';
    if (n <= 100) return '#e8533e';
    return '#c0233c';
}}

// Ray-casting point-in-polygon
function pointInPoly(lat, lon, coords) {{
    var inside = false, n = coords.length;
    for (var i = 0, j = n-1; i < n; j = i++) {{
        var xi = coords[i][0], yi = coords[i][1];
        var xj = coords[j][0], yj = coords[j][1];
        if (((yi > lon) !== (yj > lon)) &&
            (lat < (xj-xi)*(lon-yi)/(yj-yi)+xi))
            inside = !inside;
    }}
    return inside;
}}

// Pre-assign each incident to a tract (runs once on page load)
var incidentTract = new Array(SHOOT_DATA.length).fill(null);
window.addEventListener('load', function() {{
    for (var i = 0; i < SHOOT_DATA.length; i++) {{
        var pt = SHOOT_DATA[i];
        for (var t = 0; t < TRACT_POLYS.length; t++) {{
            if (pointInPoly(pt.lat, pt.lon, TRACT_POLYS[t].coords)) {{
                incidentTract[i] = TRACT_POLYS[t].id;
                break;
            }}
        }}
    }}
    setTimeout(function() {{
        for (var k in window) {{
            try {{
                if (window[k] && window[k]._leaflet_id && window[k].eachLayer)
                    window._leaflet_map = window[k];
            }} catch(e) {{}}
        }}
        console.log('[BostonMap] Ready.');
    }}, 600);
}});

function applyDateFilter() {{
    var startVal = document.getElementById('bm-start').value;
    var endVal   = document.getElementById('bm-end').value;
    if (!startVal || !endVal) {{ alert('Please select both a start and end date.'); return; }}
    var start = new Date(startVal + 'T00:00:00Z');
    var end   = new Date(endVal   + 'T23:59:59Z');
    if (start > end) {{ alert('Start date must be before end date.'); return; }}

    var tractCounts = {{}};
    var total = 0;
    for (var i = 0; i < SHOOT_DATA.length; i++) {{
        var d = new Date(SHOOT_DATA[i].date + 'T00:00:00Z');
        if (d >= start && d <= end) {{
            var tid = incidentTract[i];
            if (tid) tractCounts[tid] = (tractCounts[tid] || 0) + 1;
            total++;
        }}
    }}

    var map = window._leaflet_map;
    if (map) {{
        map.eachLayer(function(layer) {{
            if (layer.getLayers) {{
                layer.eachLayer(function(child) {{
                    if (child.feature && child.feature.properties.tract_id) {{
                        var tid = child.feature.properties.tract_id;
                        var cnt = tractCounts[tid] || 0;
                        child.setStyle({{ fillColor: countToColor(cnt),
                            fillOpacity: 0.70, color: '#bbb', weight: 0.8 }});
                        child.feature.properties.shoot_count = cnt;
                    }}
                }});
            }} else if (layer.feature && layer.feature.properties.tract_id) {{
                var tid = layer.feature.properties.tract_id;
                var cnt = tractCounts[tid] || 0;
                layer.setStyle({{ fillColor: countToColor(cnt),
                    fillOpacity: 0.70, color: '#bbb', weight: 0.8 }});
                layer.feature.properties.shoot_count = cnt;
            }}
        }});
    }}

    var fmt = function(s) {{
        return new Date(s + 'T00:00:00Z')
            .toLocaleDateString('en-US', {{month:'short', year:'numeric', timeZone:'UTC'}});
    }};
    document.getElementById('bm-title-range').textContent =
        'Boston Total Shootings ' + fmt(startVal) + ' to ' + fmt(endVal);
    document.getElementById('bm-title-n').textContent = 'N = ' + total.toLocaleString();
}}

function setYTD() {{
    var today = new Date();
    var y  = today.getFullYear();
    var mm = String(today.getMonth() + 1).padStart(2, '0');
    var dd = String(today.getDate()).padStart(2, '0');
    document.getElementById('bm-start').value = y + '-01-01';
    document.getElementById('bm-end').value   = y + '-' + mm + '-' + dd;
    applyDateFilter();
}}
</script>
"""

m.get_root().html.add_child(folium.Element(filter_js))
print("Date-filter JS injected")

Date-filter JS injected


In [10]:
# Title bar (span IDs updated by JS on filter apply)
date_min_lbl = df_shoot["shooting_date"].min().strftime("%b %Y")
date_max_lbl = df_shoot["shooting_date"].max().strftime("%b %Y")

title_html = f"""
<div id="bm-titlebar" style="
     position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:9999;
     background:rgba(255,255,255,0.94);padding:8px 20px;border-radius:4px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#222;
     font-size:16px;font-weight:600;
     box-shadow:0 1px 4px rgba(0,0,0,0.12);border:1px solid #ddd;white-space:nowrap;">
    <span id="bm-title-range">Boston Total Shootings {date_min_lbl} to {date_max_lbl}</span>
    <span id="bm-title-n" style="color:#888;font-weight:400;">&nbsp;·&nbsp;N = {TOTAL_N:,}</span>
    &nbsp;&amp;&nbsp;
    <span style="color:{CREW_ORANGE};font-weight:600;">Crews</span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# Date-filter widget (bottom-right — clear of layer toggle which sits top-right)
filter_widget_html = f"""
<div id="bm-filter-panel" style="
     position:fixed;bottom:20px;right:12px;z-index:9999;
     background:rgba(255,255,255,0.96);padding:14px 18px;border-radius:6px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#333;
     font-size:13px;line-height:1.9;
     box-shadow:0 2px 8px rgba(0,0,0,0.14);border:1px solid #ddd;min-width:220px;">

    <div style="font-size:14px;font-weight:700;margin-bottom:10px;color:#222;">Date Range Filter</div>

    <label style="display:block;font-size:12px;color:#666;margin-bottom:3px;">Start date</label>
    <input id="bm-start" type="date" value="{DATA_START}" min="{DATA_START}" max="{TODAY}"
           style="width:100%;padding:5px 8px;border:1px solid #ccc;border-radius:3px;
                  font-size:13px;box-sizing:border-box;margin-bottom:10px;">

    <label style="display:block;font-size:12px;color:#666;margin-bottom:3px;">End date</label>
    <input id="bm-end" type="date" value="{DATA_END}" min="{DATA_START}" max="{TODAY}"
           style="width:100%;padding:5px 8px;border:1px solid #ccc;border-radius:3px;
                  font-size:13px;box-sizing:border-box;margin-bottom:12px;">

    <div style="display:flex;gap:8px;">
        <button onclick="applyDateFilter()" style="
            flex:1;padding:7px 0;background:#3a7abf;color:#fff;
            border:none;border-radius:3px;font-size:13px;font-weight:600;cursor:pointer;">Apply</button>
        <button onclick="setYTD()" title="Year-to-date: Jan 1 {CUR_YEAR} to today" style="
            flex:1;padding:7px 0;background:#2d8c4e;color:#fff;
            border:none;border-radius:3px;font-size:13px;font-weight:600;cursor:pointer;">YTD</button>
    </div>
    <div style="margin-top:8px;font-size:11px;color:#999;">YTD = Jan 1 {CUR_YEAR} to today</div>
</div>
"""
m.get_root().html.add_child(folium.Element(filter_widget_html))
print("Filter widget panel added")

Filter widget panel added


In [11]:
# Legend (left side)
legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
     background:rgba(255,255,255,0.94);padding:16px 20px;border-radius:4px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#333;
     font-size:13px;line-height:1.8;max-width:260px;
     box-shadow:0 1px 4px rgba(0,0,0,0.12);border:1px solid #ddd;">

    <div style="font-size:15px;font-weight:700;margin-bottom:5px;">Boston Crews</div>
    <div style="font-size:11px;color:#666;margin-bottom:5px;">Number of Catalysts</div>
    {''.join([
        f'<div style="display:flex;align-items:center;gap:10px;margin-top:4px;">'
        f'<div style="width:{sz}px;height:{sz}px;border-radius:50%;'
        f'background:{CREW_ORANGE};border:2px solid #c45a1a;opacity:0.85;flex-shrink:0;"></div>'
        f'<span>{lbl}</span></div>'
        for sz, lbl in [(28,'>12'),(22,'9'),(16,'6'),(12,'4'),(8,'<4')]
    ])}

    <div style="margin-top:12px;padding-top:10px;border-top:1px solid #ddd;
         font-size:15px;font-weight:700;margin-bottom:5px;">Catalyst Type</div>
    <div style="display:flex;align-items:center;gap:10px;margin-top:4px;">
        <div style="width:12px;height:12px;border-radius:50%;background:{STIPEND_COLOR};flex-shrink:0;"></div>
        <span>Stipend</span></div>
    <div style="display:flex;align-items:center;gap:10px;margin-top:4px;">
        <div style="width:12px;height:12px;border-radius:50%;background:{NETWORK_COLOR};flex-shrink:0;"></div>
        <span>In-Network</span></div>

    <div style="margin-top:12px;padding-top:10px;border-top:1px solid #ddd;
         font-size:15px;font-weight:700;margin-bottom:5px;">Shootings by Census Tract</div>
    <div style="font-size:11px;color:#666;margin-bottom:5px;">Fatal &amp; Nonfatal · updates with filter</div>
    {''.join([
        f'<div style="display:flex;align-items:center;gap:8px;margin-top:3px;">'
        f'<div style="width:20px;height:14px;background:{col};border:1px solid #bbb;flex-shrink:0;"></div>'
        f'<span>{lbl}</span></div>'
        for col, lbl in [
            ('#2d8c4e','0'), ('#78b85e','1'), ('#c8d96f','2-3'),
            ('#ffe08b','4-10'), ('#fdbe6f','11-25'),
            ('#f4845f','26-50'), ('#e8533e','51-100'), ('#c0233c','100+'),
        ]
    ])}

    <div style="margin-top:12px;padding-top:10px;border-top:1px solid #ddd;">
        <div style="display:flex;align-items:center;gap:8px;margin-top:4px;">
            <svg width="22" height="4"><line x1="0" y1="2" x2="22" y2="2"
              stroke="#56B4E9" stroke-width="2.5"/></svg><span>Affiliation</span></div>
        <div style="display:flex;align-items:center;gap:8px;margin-top:4px;">
            <svg width="22" height="4"><line x1="0" y1="2" x2="22" y2="2"
              stroke="#D55E00" stroke-width="2.5" stroke-dasharray="4,3"/></svg><span>Conflict</span></div>
        <div style="display:flex;align-items:center;gap:8px;margin-top:4px;">
            <svg width="22" height="4"><line x1="0" y1="2" x2="22" y2="2"
              stroke="#888" stroke-width="1.5" stroke-dasharray="2,4"/></svg><span>Catalyst link</span></div>
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=True).add_to(m)
print("Legend and layer control added")
m

Legend and layer control added


## 7. Export

In [12]:
m.save("boston_crew_network_dashboard.html")
print("Saved -> boston_crew_network_dashboard.html")
print()
print("Date-filter widget is fully client-side -- no server needed.")
print("To refresh data: set USE_LOCAL = False and re-run all cells.")
print("Boston PD updates weekly with a 7-day delay.")
print()
print("Weekly refresh via cron + papermill:")
print("  0 6 * * MON papermill boston_crew_dashboard.ipynb out.ipynb")

Saved -> boston_crew_network_dashboard.html

Date-filter widget is fully client-side -- no server needed.
To refresh data: set USE_LOCAL = False and re-run all cells.
Boston PD updates weekly with a 7-day delay.

Weekly refresh via cron + papermill:
  0 6 * * MON papermill boston_crew_dashboard.ipynb out.ipynb
